In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import warnings
import re

warnings.filterwarnings('ignore')

CSV_PATH = "cgm_raw_data.csv"

DATE_COL = "day__date"
PERSON_COL = "day__client_id__v000"

HARD_MIN, HARD_MAX = 2.0, 25.0
SOFT_LOW, SOFT_HIGH = 3.0, 15.0
df = pd.read_csv(CSV_PATH)

time_cols = [c for c in df.columns if re.fullmatch(r"\d{2}:\d{2}", str(c))]
if not time_cols:
    raise ValueError("No time columns found matching HH:MM (e.g., '00:00').")

keep_cols = [DATE_COL, PERSON_COL] + time_cols
missing = [c for c in [DATE_COL, PERSON_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

wide = df[keep_cols].copy()
wide[DATE_COL] = pd.to_datetime(wide[DATE_COL], errors="coerce").dt.date
wide = wide.dropna(subset=[DATE_COL, PERSON_COL])

long = wide.melt(
    id_vars=[DATE_COL, PERSON_COL],
    value_vars=time_cols,
    var_name="time",
    value_name="glucose_mmol_l"
)

long["datetime"] = pd.to_datetime(
    long[DATE_COL].astype(str) + " " + long["time"],
    errors="coerce"
)

long["glucose_mmol_l"] = pd.to_numeric(long["glucose_mmol_l"], errors="coerce")
long = long.dropna(subset=["datetime", "glucose_mmol_l"])

long["is_hard_outlier"] = (long["glucose_mmol_l"] < HARD_MIN) | (long["glucose_mmol_l"] > HARD_MAX)
long = long.loc[~long["is_hard_outlier"]].copy()
long["is_soft_extreme"] = (long["glucose_mmol_l"] < SOFT_LOW) | (long["glucose_mmol_l"] > SOFT_HIGH)

long = long.rename(columns={
    PERSON_COL: "person_id",
    DATE_COL: "date"
})[["person_id", "date", "datetime", "glucose_mmol_l", "is_soft_extreme"]]

long = long.sort_values(["person_id", "datetime"]).reset_index(drop=True)

dt = long.groupby("person_id")["datetime"].diff()
median_step_min = dt.dt.total_seconds().div(60).median()
print("Rows (long):", len(long))
print("People:", long["person_id"].nunique())
print("Median step (min):", median_step_min)

per_day_counts = long.groupby(["person_id", "date"]).size()
print("Per-day rows (min/median/max):",
      int(per_day_counts.min()), int(per_day_counts.median()), int(per_day_counts.max()))

long.to_csv('long_format_data.csv', index=False)
print("Long format DataFrame preview:")
print(long.head())
print("\nDataFrame info:")
print(long.info())

# 2. SENSITIVE SPIKE DETECTION ALGORITHM


Long_CSV_PATH = "long_format_data.csv"
df = pd.read_csv(Long_CSV_PATH)
df.columns = df.columns.str.strip()


Rows (long): 60318
People: 20
Median step (min): 5.0
Per-day rows (min/median/max): 3 288 288
Long format DataFrame preview:
    person_id        date            datetime  glucose_mmol_l  is_soft_extreme
0  SPIKES-001  2026-02-24 2026-02-24 12:35:00             4.8            False
1  SPIKES-001  2026-02-24 2026-02-24 12:40:00             5.1            False
2  SPIKES-001  2026-02-24 2026-02-24 12:45:00             5.3            False
3  SPIKES-001  2026-02-24 2026-02-24 12:50:00             5.2            False
4  SPIKES-001  2026-02-24 2026-02-24 12:55:00             5.2            False

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60318 entries, 0 to 60317
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   person_id        60318 non-null  object        
 1   date             60318 non-null  object        
 2   datetime         60318 non-null  datetime64[ns]
 3   glucos

In [3]:

#  PRESETS

PRESETS_SPIKE = {
    "high_confidence": dict(
        amp_thresh=1.0,
        rise_rate_thresh=0.02,
        rise_window_min=15,
        min_above_min=15,
        end_band=0.3,
        refractory_min=30,
        gate_mode="and",
        require_return=True,
        max_event_duration_min=300,
        baseline_lookback_min=60,
        baseline_exclude_last_min=10,
    ),
    "sensitive": dict(
        amp_thresh=0.4,
        rise_rate_thresh=0.008,
        rise_window_min=20,
        min_above_min=5,
        end_band=0.5,
        refractory_min=10,
        gate_mode="or",
        require_return=False,
        max_event_duration_min=180,
        baseline_lookback_min=90,
        baseline_exclude_last_min=5,
    ),
}

PRESETS_DIP = {
    "high_confidence": dict(
        amp_thresh=0.8,            # dips are smaller amplitude typically
        rise_rate_thresh=0.015,    # fall rate
        rise_window_min=15,
        min_above_min=10,          # 10 min below threshold
        end_band=0.3,
        refractory_min=30,
        gate_mode="and",
        require_return=True,
        max_event_duration_min=300,
        baseline_lookback_min=60,
        baseline_exclude_last_min=10,
        absolute_thresh=3.9,       # clinical hypo threshold (mmol/L)
    ),
    "sensitive": dict(
        amp_thresh=0.3,            # very subtle dips
        rise_rate_thresh=0.006,
        rise_window_min=20,
        min_above_min=5,           # even brief dips matter
        end_band=0.5,
        refractory_min=10,
        gate_mode="or",
        require_return=False,
        max_event_duration_min=180,
        baseline_lookback_min=90,
        baseline_exclude_last_min=5,
        absolute_thresh=4.2,       # slightly higher floor for sensitive mode
    ),
}


# ══════════════════════════════════════════════════════════════
#  UNIFIED DETECTION FUNCTION
# ══════════════════════════════════════════════════════════════

def find_events(
    df: pd.DataFrame,
    person_id="",
    *,
    mode: str = "both",            # "high_confidence", "sensitive", "both"
    direction: str = "both",       # "up" (spikes), "down" (dips), "both"
    glucose_col: str = "glucose_mmol_l",
    dt_col: str = "datetime",
    id_col: str = "person_id",
    date_col: str = "date",
    sample_step_min: int = 5,
    # ── Confidence scoring weights ──
    w_amp: float = 0.45,
    w_slope: float = 0.20,
    w_dur: float = 0.15,
    w_return: float = 0.10,
    w_snr: float = 0.10,
    expected_noise_mad_floor: float = 0.05,
    # ── Override presets (optional — if provided, these override the preset) ──
    **override_kwargs,
):
    """
    Unified glucose event detector.

    Detects spikes (up), dips (down), or both, at high-confidence
    or sensitive thresholds, or both.

    Returns:
        events_df:    One row per event with columns including 'direction' and 'tier'
        annotated_df: Original trace with event annotations
    """

    EVENT_COLS = [
        "Day", "start_time", "end_time", "peak_time", "p_val",
        "range", "extreme_glucose", "baseline_at_extreme",
        "closure_reason", "direction", "tier",
    ]

    # ── Resolve which runs to perform ──
    directions = []
    if direction in ("up", "both"):
        directions.append("up")
    if direction in ("down", "both"):
        directions.append("down")

    modes = []
    if mode in ("high_confidence", "both"):
        modes.append("high_confidence")
    if mode in ("sensitive", "both"):
        modes.append("sensitive")

    # ── Filter + sort for this person ──
    if id_col not in df.columns:
        raise KeyError(f"'{id_col}' not found. Available columns: {df.columns.tolist()}")

    d = df.loc[df[id_col].astype(str) == str(person_id)].copy()

    if d.empty:
        return (
            pd.DataFrame(columns=EVENT_COLS),
            pd.DataFrame(columns=[glucose_col, dt_col, id_col, date_col,
                                  "baseline", "slope", "residual",
                                  "event_id", "in_event", "is_event_start",
                                  "is_event_end", "is_event_extreme",
                                  "p_val_event", "event_direction", "event_tier"]),
        )

    d[dt_col] = pd.to_datetime(d[dt_col])
    if date_col in d.columns:
        d[date_col] = pd.to_datetime(d[date_col]).dt.date
    else:
        d[date_col] = d[dt_col].dt.date

    d = d.sort_values(dt_col).reset_index(drop=True)
    d = d.dropna(subset=[glucose_col]).reset_index(drop=True)

    g = d[glucose_col].to_numpy(dtype=float)
    t = d[dt_col].to_numpy()
    day_arr = d[date_col].to_numpy()
    n = len(g)
    idx_per_min = 1.0 / sample_step_min

    # ── Shared computations (done once) ──
    # Noise estimate
    smooth = pd.Series(g).rolling(window=7, center=True, min_periods=3).median().to_numpy()
    resid_short = g - smooth
    mad = np.nanmedian(np.abs(resid_short - np.nanmedian(resid_short)))
    noise = max(1.4826 * mad, expected_noise_mad_floor)

    # ── Run each (direction × mode) combination ──
    all_events = []

    for dirn in directions:
        for m in modes:
            # Select preset
            if dirn == "up":
                params = PRESETS_SPIKE[m].copy()
            else:
                params = PRESETS_DIP[m].copy()

            # Apply any user overrides
            params.update(override_kwargs)

            events = _detect_single_direction(
                g=g, t=t, day_arr=day_arr, n=n,
                idx_per_min=idx_per_min,
                noise=noise,
                dirn=dirn,
                sample_step_min=sample_step_min,
                w_amp=w_amp, w_slope=w_slope, w_dur=w_dur,
                w_return=w_return, w_snr=w_snr,
                **params,
            )

            for ev in events:
                ev["direction"] = dirn
                ev["tier"] = m

            all_events.extend(events)

    # ── Deduplicate overlapping events ──
    events_df = pd.DataFrame(all_events, columns=EVENT_COLS)
    if not events_df.empty:
        events_df = _deduplicate_events(events_df)
        events_df = events_df.sort_values(["Day", "start_time"]).reset_index(drop=True)

    # ── Build annotated trace ──
    # Recompute baseline with the most generous lookback for annotation
    max_lookback = 90
    max_excl = 5
    lookback_n = int(max_lookback * idx_per_min)
    excl_n = int(max_excl * idx_per_min)

    B = np.full(n, np.nan)
    for i in range(n):
        s = i - lookback_n
        e = i - excl_n
        if s >= 0 and e > s:
            window = g[s:e]
            valid = window[np.isfinite(window)]
            if len(valid) >= 3:
                B[i] = np.nanmedian(valid)

    first_valid = np.where(np.isfinite(B))[0]
    if len(first_valid) > 0:
        B[:first_valid[0]] = B[first_valid[0]]

    # Slope (20-min window)
    rise_n = int(20 * idx_per_min)
    slope = np.full(n, np.nan)
    for i in range(n):
        j = i - rise_n
        if j >= 0:
            slope[i] = (g[i] - g[j]) / 20.0

    annotated = d.copy()
    annotated["baseline"] = B
    annotated["slope"] = slope
    annotated["residual"] = g - B
    annotated["event_id"] = pd.Series([pd.NA] * n, dtype="Int64")
    annotated["in_event"] = False
    annotated["is_event_start"] = False
    annotated["is_event_end"] = False
    annotated["is_event_extreme"] = False
    annotated["p_val_event"] = np.nan
    annotated["event_direction"] = ""
    annotated["event_tier"] = ""

    # Map events back onto the trace
    if not events_df.empty:
        for eid, row in events_df.iterrows():
            st = pd.to_datetime(row["start_time"])
            et = pd.to_datetime(row["end_time"])
            pt = pd.to_datetime(row["peak_time"])

            mask = (annotated[dt_col] >= st) & (annotated[dt_col] <= et)
            annotated.loc[mask, "event_id"] = eid
            annotated.loc[mask, "in_event"] = True
            annotated.loc[mask, "p_val_event"] = row["p_val"]
            annotated.loc[mask, "event_direction"] = row["direction"]
            annotated.loc[mask, "event_tier"] = row["tier"]

            # Start
            start_mask = mask & (annotated[dt_col] == st)
            annotated.loc[start_mask, "is_event_start"] = True

            # End
            end_mask = mask & (annotated[dt_col] == et)
            annotated.loc[end_mask, "is_event_end"] = True

            # Extreme (peak or trough)
            extreme_mask = mask & ((annotated[dt_col] - pt).abs() ==
                                   (annotated[dt_col] - pt).abs().min())
            annotated.loc[extreme_mask, "is_event_extreme"] = True

    return events_df, annotated


def _detect_single_direction(
    g, t, day_arr, n, idx_per_min, noise, dirn,
    *,
    amp_thresh, rise_rate_thresh, rise_window_min, min_above_min,
    end_band, refractory_min, gate_mode, require_return,
    max_event_duration_min, baseline_lookback_min,
    baseline_exclude_last_min, sample_step_min,
    w_amp, w_slope, w_dur, w_return, w_snr,
    absolute_thresh=None,
    **_extra,
):
    """
    Core detection logic for a single direction (up or down).
    For dips, we invert the signal, run the same logic, then un-invert.
    """

    # ── For dips: negate the signal so we can reuse spike logic ──
    if dirn == "down":
        g_work = -g.copy()
    else:
        g_work = g.copy()

    lookback_n = int(baseline_lookback_min * idx_per_min)
    excl_n = int(baseline_exclude_last_min * idx_per_min)
    rise_n = int(rise_window_min * idx_per_min)
    min_above_n = max(1, int(min_above_min * idx_per_min))
    refractory_n = int(refractory_min * idx_per_min)
    max_event_n = int(max_event_duration_min * idx_per_min)

    if n < (lookback_n + rise_n + 1):
        return []

    # ── Baseline ──
    B = np.full(n, np.nan)
    for i in range(n):
        s = i - lookback_n
        e = i - excl_n
        if s >= 0 and e > s:
            window = g_work[s:e]
            valid = window[np.isfinite(window)]
            if len(valid) >= 3:
                B[i] = np.nanmedian(valid)

    first_valid = np.where(np.isfinite(B))[0]
    if len(first_valid) > 0:
        B[:first_valid[0]] = B[first_valid[0]]

    # ── Slope ──
    slope = np.full(n, np.nan)
    for i in range(n):
        j = i - rise_n
        if j >= 0:
            slope[i] = (g_work[i] - g_work[j]) / rise_window_min

    # ── Detection loop ──
    events = []
    i = 0
    in_refractory_until = -1

    while i < n:
        if i < in_refractory_until:
            if not (np.isfinite(B[i]) and g_work[i] <= B[i] + end_band):
                i += 1
                continue

        if not np.isfinite(B[i]):
            i += 1
            continue

        # ── Entry gate ──
        amp_ok = (g_work[i] - B[i]) >= amp_thresh
        slope_ok = np.isfinite(slope[i]) and slope[i] >= rise_rate_thresh

        # For dips: also trigger if absolute glucose crosses clinical threshold
        if dirn == "down" and absolute_thresh is not None:
            # g_work is negated, so original glucose = -g_work
            original_glucose = -g_work[i]
            absolute_ok = original_glucose <= absolute_thresh
        else:
            absolute_ok = False

        if gate_mode == "or":
            triggered = amp_ok or slope_ok or absolute_ok
        else:
            triggered = (amp_ok and slope_ok) or absolute_ok

        if not triggered:
            i += 1
            continue

        # ── Walk back ──
        walkback_thresh = amp_thresh * 0.5
        start_i = i
        while (start_i - 1 >= 0
               and np.isfinite(B[start_i - 1])
               and (g_work[start_i - 1] - B[start_i - 1]) >= walkback_thresh):
            start_i -= 1

        # ── Minimum duration ──
        above_count = 0
        k = start_i
        while k < n and np.isfinite(B[k]) and (g_work[k] - B[k]) >= amp_thresh * 0.75:
            above_count += 1
            k += 1
            if above_count >= min_above_n:
                break

        if above_count < min_above_n:
            i += 1
            continue

        # ── Scan forward ──
        peak_i = i
        max_g = g_work[i]
        min_g = g_work[i]
        j = i
        closure_reason = "end_of_data"
        end_i = n - 1

        while j < n:
            if g_work[j] > max_g:
                max_g = g_work[j]
                peak_i = j
            if g_work[j] < min_g:
                min_g = g_work[j]

            if np.isfinite(B[j]) and g_work[j] <= B[j] + end_band:
                end_i = j
                closure_reason = "return_to_baseline"
                break

            if (j - start_i) >= max_event_n:
                end_i = j
                closure_reason = "timeout"
                break

            j += 1

        if require_return and closure_reason != "return_to_baseline":
            i += 1
            continue

        event_range = float(max_g - min_g)

        # ── Confidence score ──
        peak_amp = float(max_g - B[peak_i]) if np.isfinite(B[peak_i]) else float(max_g - np.nanmedian(B))
        amp_excess = max(0.0, peak_amp - amp_thresh)
        amp_score = 1.0 - np.exp(-(amp_excess / (noise + 1e-9)))

        slope_window_end = min(end_i, start_i + int(30 * idx_per_min))
        if slope_window_end >= start_i:
            local_slopes = slope[start_i:slope_window_end + 1]
            local_max_slope = np.nanmax(local_slopes) if np.any(np.isfinite(local_slopes)) else 0.0
        else:
            local_max_slope = slope[i] if np.isfinite(slope[i]) else 0.0
        slope_excess = max(0.0, float(local_max_slope) - rise_rate_thresh)
        slope_score = 1.0 - np.exp(-(slope_excess / (noise / max(rise_window_min, 1) + 1e-9)))

        above = 0
        for q in range(start_i, end_i + 1):
            if np.isfinite(B[q]) and (g_work[q] - B[q]) >= amp_thresh * 0.75:
                above += 1
        dur_min = above * sample_step_min
        dur_score = min(1.0, dur_min / 45.0)

        end_gap = float(g_work[end_i] - (B[end_i] if np.isfinite(B[end_i]) else np.nanmedian(B)))
        if closure_reason == "return_to_baseline" and np.isfinite(end_gap):
            return_score = float(np.clip(1.0 - (max(0.0, end_gap) / max(end_band, 1e-9)), 0.0, 1.0))
        elif closure_reason == "timeout":
            return_score = 0.25
        else:
            return_score = 0.15

        snr = peak_amp / (noise + 1e-9)
        snr_score = float(np.clip(snr / 5.0, 0.0, 1.0))

        p_val = float(np.clip(
            w_amp * amp_score + w_slope * slope_score + w_dur * dur_score
            + w_return * return_score + w_snr * snr_score,
            0.0, 1.0,
        ))

        # ── Un-invert values for dips ──
        if dirn == "down":
            extreme_glucose = float(-max_g)       # the trough (lowest real glucose)
            baseline_at_extreme = float(-B[peak_i]) if np.isfinite(B[peak_i]) else np.nan
        else:
            extreme_glucose = float(max_g)         # the peak (highest real glucose)
            baseline_at_extreme = float(B[peak_i]) if np.isfinite(B[peak_i]) else np.nan

        events.append({
            "Day": day_arr[start_i],
            "start_time": pd.to_datetime(t[start_i]),
            "end_time": pd.to_datetime(t[end_i]),
            "peak_time": pd.to_datetime(t[peak_i]),
            "p_val": round(p_val, 4),
            "range": round(event_range, 3),
            "extreme_glucose": round(extreme_glucose, 3),
            "baseline_at_extreme": round(baseline_at_extreme, 3) if not np.isnan(baseline_at_extreme) else np.nan,
            "closure_reason": closure_reason,
        })

        in_refractory_until = end_i + refractory_n
        i = end_i + 1

    return events


def _deduplicate_events(events_df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove overlapping events, keeping the higher-tier version.
    Priority: high_confidence > sensitive (within same direction).
    For cross-direction overlaps (spike overlapping a dip), keep both.
    """
    if events_df.empty:
        return events_df

    tier_priority = {"high_confidence": 0, "sensitive": 1}
    events_df["_priority"] = events_df["tier"].map(tier_priority).fillna(2)
    events_df = events_df.sort_values(["direction", "_priority", "start_time"]).reset_index(drop=True)

    keep = []
    for dirn in events_df["direction"].unique():
        dir_events = events_df[events_df["direction"] == dirn].copy()
        dir_events = dir_events.sort_values(["_priority", "start_time"]).reset_index(drop=True)

        kept_intervals = []  # list of (start, end) already kept

        for _, row in dir_events.iterrows():
            st = pd.to_datetime(row["start_time"])
            et = pd.to_datetime(row["end_time"])

            overlaps = False
            for ks, ke in kept_intervals:
                if st <= ke and et >= ks:
                    overlaps = True
                    break

            if not overlaps:
                keep.append(row)
                kept_intervals.append((st, et))

    result = pd.DataFrame(keep)
    if "_priority" in result.columns:
        result = result.drop(columns=["_priority"])

    return result.reset_index(drop=True)


# ══════════════════════════════════════════════════════════════
#  BATCH RUNNER
# ══════════════════════════════════════════════════════════════

def run_all_events(
    df,
    *,
    id_col="person_id",
    mode="both",
    direction="both",
    output_events_csv="events_all_clients.csv",
    output_annotated_csv="cgm_annotated_all_clients.csv",
    **kwargs,
):
    df = df.copy()
    df.columns = df.columns.str.strip()

    person_ids = df[id_col].dropna().astype(str).unique()
    all_events = []
    all_annotated = []

    for pid in person_ids:
        ev, ann = find_events(df, person_id=pid, mode=mode,
                              direction=direction, id_col=id_col, **kwargs)
        ev.insert(0, id_col, pid)
        ann[id_col] = pid
        all_events.append(ev)
        all_annotated.append(ann)

    events_all = pd.concat(all_events, ignore_index=True)
    annotated_all = pd.concat(all_annotated, ignore_index=True)

    events_all.to_csv(output_events_csv, index=False)
    annotated_all.to_csv(output_annotated_csv, index=False)

    return events_all, annotated_all


# ══════════════════════════════════════════════════════════════
#  VISUALISATION — DAILY TRACE WITH SPIKES & DIPS
# ══════════════════════════════════════════════════════════════

def plot_events_daily(
    annotated_df: pd.DataFrame,
    events_df: pd.DataFrame,
    person_id: str,
    out_dir: str = "event_plots",
    glucose_col: str = "glucose_mmol_l",
    dt_col: str = "datetime",
    day_col: str = "date",
    dpi: int = 180,
):
    """
    One PNG per day. Spikes and dips shaded in different colours,
    tiered by confidence.
    """

    os.makedirs(out_dir, exist_ok=True)

    a = annotated_df.copy()
    a[dt_col] = pd.to_datetime(a[dt_col])
    if day_col in a.columns:
        a[day_col] = pd.to_datetime(a[day_col]).dt.date
    else:
        a[day_col] = a[dt_col].dt.date

    ev = events_df.copy()
    if not ev.empty:
        for col in ["start_time", "end_time", "peak_time"]:
            if col in ev.columns:
                ev[col] = pd.to_datetime(ev[col])
        if "Day" in ev.columns:
            ev["Day"] = pd.to_datetime(ev["Day"]).dt.date

    # ── Colour scheme ──
    EVENT_COLORS = {
        ("up", "high_confidence"):   {"shade": "#FF4500", "alpha": 0.30, "marker": "red",       "label": "Spike (HC)"},
        ("up", "sensitive"):         {"shade": "#FFA500", "alpha": 0.20, "marker": "orange",     "label": "Spike (Sens)"},
        ("down", "high_confidence"): {"shade": "#4169E1", "alpha": 0.30, "marker": "blue",       "label": "Dip (HC)"},
        ("down", "sensitive"):       {"shade": "#87CEEB", "alpha": 0.20, "marker": "lightskyblue","label": "Dip (Sens)"},
    }

    for day in sorted(a[day_col].dropna().unique()):
        day_df = a.loc[a[day_col] == day].sort_values(dt_col)

        fig, ax = plt.subplots(figsize=(16, 5.5))

        # Glucose trace
        ax.plot(day_df[dt_col], day_df[glucose_col],
                color="black", lw=1.3, zorder=3, label="Glucose")

        # Baseline
        if "baseline" in day_df.columns:
            ax.plot(day_df[dt_col], day_df["baseline"],
                    color="gray", lw=1.0, alpha=0.6, ls="--", label="Baseline")

        # Clinical hypo line
        ax.axhline(y=3.9, color="blue", ls=":", lw=0.8, alpha=0.5, label="Hypo threshold (3.9)")

        # Shade events and mark extremes
        legend_added = set()

        if not ev.empty:
            ev_day = ev.loc[ev["Day"] == day] if "Day" in ev.columns else ev.loc[ev["start_time"].dt.date == day]

            for _, e in ev_day.iterrows():
                dirn = e.get("direction", "up")
                tier = e.get("tier", "sensitive")
                key = (dirn, tier)
                style = EVENT_COLORS.get(key, EVENT_COLORS[("up", "sensitive")])

                # Shade
                ax.axvspan(e["start_time"], e["end_time"],
                           color=style["shade"], alpha=style["alpha"], zorder=1)

                # Peak/trough marker
                if "peak_time" in e:
                    pt = e["peak_time"]
                    nearest = day_df.loc[(day_df[dt_col] - pt).abs().idxmin()]

                    marker_shape = "v" if dirn == "down" else "^"
                    marker_size = 70 if tier == "high_confidence" else 45

                    lbl = style["label"] if key not in legend_added else None
                    legend_added.add(key)

                    ax.scatter(nearest[dt_col], nearest[glucose_col],
                               color=style["marker"], marker=marker_shape,
                               s=marker_size, zorder=5,
                               edgecolors="black", linewidths=0.6, label=lbl)

                    # Annotate
                    offset_y = -22 if dirn == "down" else 18
                    closure_short = str(e.get("closure_reason", ""))[:3]
                    ax.annotate(
                        f"p={e['p_val']:.2f}",
                        xy=(nearest[dt_col], nearest[glucose_col]),
                        xytext=(0, offset_y),
                        textcoords="offset points",
                        fontsize=7, ha="center", fontweight="bold",
                        color="darkred" if dirn == "up" else "darkblue",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white",
                                  ec="gray", alpha=0.8),
                    )

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        plt.xticks(rotation=45)

        ax.set_title(f"{person_id} — {day}", fontsize=13, fontweight="bold")
        ax.set_xlabel("Time of Day", fontsize=10)
        ax.set_ylabel("Glucose (mmol/L)", fontsize=10)
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9, ncol=2)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{person_id}_{day}_events.png"), dpi=dpi)
        plt.close(fig)

    print(f"[Daily plots] Saved to: {out_dir}/")


# ══════════════════════════════════════════════════════════════
#  SUMMARY VISUALISATIONS
# ══════════════════════════════════════════════════════════════

def plot_events_summary(
    events_df: pd.DataFrame,
    annotated_df: pd.DataFrame,
    person_id: str,
    out_dir: str = "event_plots",
    glucose_col: str = "glucose_mmol_l",
    dt_col: str = "datetime",
    dpi: int = 180,
):
    """
    Summary charts: confidence distributions, timing heatmaps,
    event counts — split by direction.
    """

    os.makedirs(out_dir, exist_ok=True)
    ev = events_df.copy()

    if ev.empty:
        print(f"No events for {person_id} — skipping summary.")
        return

    ev["start_time"] = pd.to_datetime(ev["start_time"])
    ev["end_time"] = pd.to_datetime(ev["end_time"])
    ev["peak_time"] = pd.to_datetime(ev["peak_time"])
    if "Day" in ev.columns:
        ev["Day"] = pd.to_datetime(ev["Day"])

    spike_color = "#FF6347"
    dip_color = "#4682B4"

    # ── 1. Confidence distribution by direction ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, dirn, color, title in [
        (axes[0], "up", spike_color, "Spikes (↑)"),
        (axes[1], "down", dip_color, "Dips (↓)"),
    ]:
        subset = ev[ev["direction"] == dirn]
        if not subset.empty:
            sns.histplot(subset["p_val"], bins=15, kde=True, color=color,
                         edgecolor="black", ax=ax)
            ax.axvline(subset["p_val"].median(), color="black", ls="--",
                       label=f"Median = {subset['p_val'].median():.2f}")
            ax.legend()
        ax.set_title(f"{title} — {len(subset)} events", fontweight="bold")
        ax.set_xlabel("Confidence (p_val)")
        ax.set_xlim(0, 1)

    fig.suptitle(f"{person_id} — Confidence Distributions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{person_id}_confidence_by_direction.png"), dpi=dpi)
    plt.close(fig)

    # ── 2. Events per day stacked by direction ──
    if "Day" in ev.columns:
        pivot = ev.groupby(["Day", "direction"]).size().unstack(fill_value=0)
        for col in ["up", "down"]:
            if col not in pivot.columns:
                pivot[col] = 0

        fig, ax = plt.subplots(figsize=(14, 5))
        pivot[["up", "down"]].plot(
            kind="bar", stacked=True, ax=ax,
            color=[spike_color, dip_color], edgecolor="black", width=0.8,
        )
        ax.set_title(f"{person_id} — Events Per Day", fontweight="bold")
        ax.set_xlabel("Date")
        ax.set_ylabel("Event Count")
        ax.legend(["Spikes ↑", "Dips ↓"], loc="upper right")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{person_id}_events_per_day.png"), dpi=dpi)
        plt.close(fig)

    # ── 3. Timing heatmap (hour × direction) ──
    ev["peak_hour"] = ev["peak_time"].dt.hour

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, dirn, cmap, title in [
        (axes[0], "up", "YlOrRd", "Spike Timing"),
        (axes[1], "down", "YlGnBu", "Dip Timing"),
    ]:
        subset = ev[ev["direction"] == dirn]
        if not subset.empty and "Day" in subset.columns:
            hm = subset.groupby(["Day", "peak_hour"]).size().unstack(fill_value=0)
            for h in range(24):
                if h not in hm.columns:
                    hm[h] = 0
            hm = hm[sorted(hm.columns)]
            sns.heatmap(hm, cmap=cmap, annot=True, fmt="d",
                        linewidths=0.5, ax=ax, cbar_kws={"label": "Count"})
        ax.set_title(f"{title}", fontweight="bold")
        ax.set_xlabel("Hour of Day")
        ax.set_ylabel("Date")

    fig.suptitle(f"{person_id} — Event Timing Heatmaps", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{person_id}_timing_heatmap.png"), dpi=dpi)
    plt.close(fig)

    # ── 4. Tier breakdown (HC vs Sensitive) by direction ──
    fig, ax = plt.subplots(figsize=(8, 5))
    tier_counts = ev.groupby(["direction", "tier"]).size().unstack(fill_value=0)

    tier_colors = {"high_confidence": "#2E8B57", "sensitive": "#DAA520"}
    tier_counts.plot(kind="bar", ax=ax,
                     color=[tier_colors.get(c, "gray") for c in tier_counts.columns],
                     edgecolor="black", width=0.6)
    ax.set_title(f"{person_id} — Detection Tier Breakdown", fontweight="bold")
    ax.set_xlabel("Direction")
    ax.set_ylabel("Event Count")
    ax.set_xticklabels(["Dips ↓", "Spikes ↑"], rotation=0)
    ax.legend(title="Tier")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{person_id}_tier_breakdown.png"), dpi=dpi)
    plt.close(fig)

    # ── 5. Confidence vs amplitude coloured by direction ──
    if "baseline_at_extreme" in ev.columns:
        ev["amplitude"] = (ev["extreme_glucose"] - ev["baseline_at_extreme"]).abs()

        fig, ax = plt.subplots(figsize=(9, 6))
        for dirn, color, marker, label in [
            ("up", spike_color, "^", "Spike ↑"),
            ("down", dip_color, "v", "Dip ↓"),
        ]:
            subset = ev[ev["direction"] == dirn]
            if not subset.empty:
                # Size by tier
                sizes = subset["tier"].map({"high_confidence": 80, "sensitive": 40}).fillna(40)
                ax.scatter(subset["amplitude"], subset["p_val"],
                           c=color, marker=marker, s=sizes,
                           edgecolors="black", linewidths=0.5,
                           label=label, alpha=0.8)

        ax.set_title(f"{person_id} — Confidence vs Amplitude", fontweight="bold")
        ax.set_xlabel("Amplitude (mmol/L)")
        ax.set_ylabel("Confidence (p_val)")
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{person_id}_confidence_vs_amplitude.png"), dpi=dpi)
        plt.close(fig)

    print(f"[Summary plots] Saved to: {out_dir}/")


# ══════════════════════════════════════════════════════════════
#  RUN EVERYTHING
# ══════════════════════════════════════════════════════════════

# Single person demo
pid = "SPIKES-001"
events_df, annotated_df = find_events(df, person_id=pid, mode="both", direction="both")

print(f"\n{'='*60}")
print(f"UNIFIED PIPELINE — {pid}")
print(f"{'='*60}")
print(f"Total events: {len(events_df)}")

if not events_df.empty:
    print(f"\nBreakdown:")
    print(events_df.groupby(["direction", "tier"]).size().to_string())
    print(f"\nEvents table:")
    print(events_df.to_string(index=False))

plot_events_daily(annotated_df, events_df, person_id=pid, out_dir=f"event_plots/{pid}")
plot_events_summary(events_df, annotated_df, person_id=pid, out_dir=f"event_plots/{pid}")

# All persons
print(f"\n{'='*60}")
print("RUNNING FOR ALL PERSONS")
print(f"{'='*60}")

events_all, annotated_all = run_all_events(
    df, mode="both", direction="both",
    output_events_csv="events_all_clients.csv",
    output_annotated_csv="cgm_annotated_all_clients.csv",
)

print(f"\nTotal events across all clients: {len(events_all)}")
print(events_all.groupby(["direction", "tier"]).size())

# Per-person plots
for pid in df["person_id"].dropna().astype(str).unique():
    ev = events_all[events_all["person_id"] == pid]
    ann = annotated_all[annotated_all["person_id"] == pid]
    plot_events_daily(ann, ev, person_id=pid, out_dir=f"event_plots/{pid}")
    plot_events_summary(ev, ann, person_id=pid, out_dir=f"event_plots/{pid}")

print("\nAll done.")


UNIFIED PIPELINE — SPIKES-001
Total events: 148

Breakdown:
direction  tier           
down       high_confidence    28
           sensitive          62
up         high_confidence    13
           sensitive          45

Events table:
       Day          start_time            end_time           peak_time  p_val  range  extreme_glucose  baseline_at_extreme     closure_reason direction            tier
2026-02-24 2026-02-24 12:35:00 2026-02-24 12:35:00 2026-02-24 12:35:00 0.5584    0.0              4.8                 5.30 return_to_baseline      down       sensitive
2026-02-24 2026-02-24 13:55:00 2026-02-24 13:55:00 2026-02-24 13:55:00 0.3364    0.0              5.7                 5.30 return_to_baseline        up       sensitive
2026-02-24 2026-02-24 15:00:00 2026-02-24 15:00:00 2026-02-24 15:00:00 0.7250    0.0              5.7                 6.10 return_to_baseline      down       sensitive
2026-02-24 2026-02-24 17:45:00 2026-02-24 17:45:00 2026-02-24 17:45:00 0.7258    0.0         